# SDA with general observation operators

`obs.assimilate` (notebook 01) observes variables on pixel masks. Many real
products are not like that: a coarse gridded field is an *average* over
many model cells, an altimeter samples *between* grid points and *between*
frame times, its errors are *correlated* along track, and a radiometer
footprint averages over an area and sometimes over time. All of these are
still linear in the state, so SDA handles them with the same prior; only the
observation operator `H` and the error covariance `R` change.

This notebook uses the opt-in general interface:

* `obottle_starter.inference.sda.operators`: `block_average`,
  `point_observations`, `footprint_observations` (and `+`/`*` to combine
  them), `ObservationSet`, and `ObservationCovariance` /
  `exponential_covariance` for correlated errors;
* `obottle_starter.inference.general.assimilate_general`: physical units in
  and out, solved in model units with a consistent scaling of `y`, `H` and
  `R` (see its module docstring).

Three experiments on one OSSE window of the released joint SSH + SST prior:

1. **Coarse-to-fine:** 64 km block averages of SSH -> 4 km SSH, against a
   simple interpolation of the coarse field.
2. **Along-track points with correlated errors:** off-grid SSH samples at
   times between frames; the right `R` against wrongly assuming independent
   errors.
3. **Footprints and time averages:** SST footprints averaged over two frames,
   assimilated together with SSH tracks.

It needs a GPU (each posterior takes a few minutes on an L40S). One window
is one case: these are demonstrations, not evaluations.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.ndimage import zoom

import obottle_starter as obs
from obottle_starter.inference.general import assimilate_general
from obottle_starter.inference.metrics import isotropic_psd, rmse
from obottle_starter.inference.osse import load_window
from obottle_starter.inference.sda.operators import (
    ObservationCovariance,
    ObservationSet,
    block_average,
    exponential_covariance,
    footprint_observations,
    gaussian_kernel,
    point_observations,
)

BUNDLE = Path("..")  # contains checkpoints/ and osse_tiles/
CKPT = BUNDLE / "checkpoints" / "joint_ssh_sst_T5.pt"
TILE = BUNDLE / "osse_tiles" / "MidAtlantic_35N_50W.nc"
device = "cuda" if torch.cuda.is_available() else "cpu"

net, cfg = obs.load_checkpoint(CKPT, device)
names = obs.channel_names(cfg)  # ['SSH', 'SST']
win = load_window(TILE, names, cfg.time_length)
shape = (len(names), cfg.time_length, cfg.H, cfg.W)  # (C, T, H, W) the operators observe
SSH, SST = names.index("SSH"), names.index("SST")
DX_KM = 4.0
common = dict(times=win.times, center_lon=win.center_lon, truth=win.truth,
              n_members=8, n_batch=4, steps=64, mmps_iterations=4, seed=0,
              verbose=False)

## 1. Coarse-to-fine: SSH from 64 km averages

`block_average(shape, SSH, 16)` is the operator of a product whose cells are
16 x 16 model cells (64 km): one observation per coarse cell and frame,
equal to the mean of the fine cells it covers. That is how the coarse
values are *formed*; interpolating them back to 4 km is not a measurement
operator, and the prior has to supply the missing fine scales.

`R` should include representation error (what a 64 km average cannot
say about a 4 km model), not only instrument noise. Here the observations
are noise-free averages of the truth, and we assume 3 mm.

In [ ]:
coarse = block_average(shape, SSH, 16, name="SSH_64km")
R_coarse = torch.full((coarse.n_obs,), 0.003**2)  # m^2
post = assimilate_general(net, cfg, coarse, R_coarse, instrument_noise=False, **common)

# Baseline: the coarse field, interpolated (cubic) back to the fine grid.
coarse_field = post.y.values.reshape(cfg.time_length, 8, 8)
interp = np.stack([zoom(f, 16, order=3, mode="nearest") for f in coarse_field])
truth = win.truth["SSH"]
mean = post.samples.sel(channel="SSH").mean("sample").values
print(f"coarse observations: {coarse.n_obs}, misfit {float(post.misfit.mean()):.2f} (whitened RMS)")
print(f"SSH RMSE  interpolation {rmse(interp, truth).mean():.4f} m | "
      f"SDA mean {rmse(mean, truth).mean():.4f} m | truth std {truth.std():.4f} m")

In [ ]:
t = 2
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
lims = dict(vmin=float(truth[t].min()), vmax=float(truth[t].max()), cmap="RdBu_r", origin="lower")
for ax, (f, title) in zip(axes, [(truth[t], "truth"), (np.kron(coarse_field[t], np.ones((16, 16))), "64 km averages (y)"),
                                 (interp[t], "cubic interpolation"), (post.samples.sel(channel="SSH")[0, t], "SDA member 0")]):
    ax.imshow(f, **lims); ax.set_title(f"SSH {title}")
plt.tight_layout()

fig, ax = plt.subplots(figsize=(6, 4))
for f, label in [(truth, "truth"), (interp, "interpolation"),
                 (post.samples.sel(channel="SSH").values, "SDA members")]:
    k, p = isotropic_psd(f, dx_km=DX_KM); ax.loglog(k, p, label=label)
ax.axvline(1 / 64, color="k", lw=0.5, ls="--"); ax.set_xlabel("wavenumber (cycles/km)"); ax.legend()
ax.set_title("SSH spectra: the dashed line is the 64 km cell size");

Interpolation has almost no power below 64 km; the members should carry
fine-scale variance from the prior while matching the coarse averages.
Whether that fine structure is *right* (not just plausible) is what the RMSE
and, over many windows, the spread-error comparison tell you.

## 2. Along-track SSH: off-grid points, between frames, correlated errors

Two synthetic tracks cross the window at different angles, sampled every
half cell (2 km). Each track is a single pass at a time *between* model
frames (fractional `frame`), so the operator interpolates in space
(bilinear) and in time (linear between the two neighbouring frames).

Along-track errors are often correlated (e.g. residual orbit or roll
errors). We generate the observations with an exponential covariance (1 cm,
40 km correlation length) plus 3 mm white noise, then assimilate them twice:
with that `R`, and wrongly assuming the same variances but independent.

In [ ]:
def track(angle_deg, frame, offset_cells, step=0.5):
    # Points along a straight line through the window centre (grid units).
    s = np.arange(-120, 120, step)
    a = np.deg2rad(angle_deg)
    rows = 63.5 + offset_cells * np.cos(a) + s * np.sin(a)
    cols = 63.5 - offset_cells * np.sin(a) + s * np.cos(a)
    keep = (rows >= 0) & (rows <= cfg.H - 1) & (cols >= 0) & (cols <= cfg.W - 1)
    return np.full(keep.sum(), float(frame)), rows[keep], cols[keep], s[keep] * DX_KM

tracks = [track(70, 0.6, -20), track(-65, 2.4, 25)]
parts, R_true, R_indep = [], [], []
for k, (f, r, c, along_km) in enumerate(tracks):
    parts.append(point_observations(shape, SSH, f, r, c, name=f"track{k}",
                                    coords={"along_km": along_km}))
    block = exponential_covariance(along_km, sigma=0.01, length=40.0, nugget=0.003)
    R_true.append(block)
    R_indep.append(block.diagonal())  # same variances, correlations dropped
tracks_op = ObservationSet(parts)
print("track observations:", tracks_op.sizes)

# Same synthetic observations (same seed) for both runs; noise drawn from the true R.
right = assimilate_general(net, cfg, tracks_op, R_true, instrument_noise=True, obs_seed=1, **common)
wrong = assimilate_general(net, cfg, tracks_op, R_indep,
                           instrument_noise=ObservationCovariance(R_true), obs_seed=1, **common)
assert np.allclose(right.y, wrong.y)

In [ ]:
from obottle_starter.inference.metrics import spread_skill

for label, ds in [("correlated R (right)", right), ("independent R (wrong)", wrong)]:
    s = ds.samples.sel(channel="SSH").values
    ss = spread_skill(s, truth)
    print(f"{label:24s} SSH RMSE {ss['rmse']:.4f} m, spread {ss['spread']:.4f} m, "
          f"error/spread {ss['ratio']:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
k0 = right.obs_group.values == 0
axes[0].plot(right.obs_along_km[k0], right.y[k0], ".", ms=2, label="observed y")
axes[0].plot(right.obs_along_km[k0], right.y_true[k0], "k", lw=1, label="truth along track")
axes[0].plot(right.obs_along_km[k0], right.predicted[0, k0], lw=1, label="member 0 (right R)")
axes[0].set_xlabel("along-track distance (km)"); axes[0].legend(); axes[0].set_title("track 0")
for ax, (ds, title) in zip(axes[1:], [(right, "right R"), (wrong, "independent R")]):
    err = ds.samples.sel(channel="SSH").mean("sample").values[2] - truth[2]
    im = ax.imshow(err, cmap="RdBu_r", vmin=-0.15, vmax=0.15, origin="lower")
    plt.colorbar(im, ax=ax, shrink=0.8, label="m")
    for f, r, c, _ in tracks: ax.plot(c, r, "k", lw=0.5)
    ax.set_title(f"ensemble-mean error, frame 2 ({title})")
plt.tight_layout()

Treating correlated errors as independent tells the sampler it has many more
independent pieces of information than it has. In principle that makes the
posterior over-confident and lets it fit a smooth along-track error as if it
were signal; the correlated `R` can attribute long-wavelength along-track
errors to error instead. **In this window the two runs barely differ**
(when we ran it: SSH RMSE 0.0569 m with the right `R` against 0.0564 m,
error/spread 1.32 against 1.29). Two tracks constrain little of the window,
and one case cannot show a difference of this size. Repeat the comparison
over many windows, and with larger correlated errors, before concluding
either way.

## 3. Footprints and time averages: SST, together with SSH tracks

A radiometer-like SST product: Gaussian footprints (sigma = 2 cells, 8 km)
on a 12-cell grid, each value the *mean over frames 0 and 1* (built by
adding two footprint operators and halving). It is assimilated jointly with
the SSH tracks above, with a separate `R` per part.

In [ ]:
rr, cc = np.meshgrid(np.arange(6, cfg.H, 12), np.arange(6, cfg.W, 12), indexing="ij")
rr, cc = rr.ravel(), cc.ravel()
kern = gaussian_kernel(2.0)
sst_mean = 0.5 * (footprint_observations(shape, SST, np.zeros_like(rr), rr, cc, kern)
                  + footprint_observations(shape, SST, np.ones_like(rr), rr, cc, kern))
sst_mean = sst_mean.renamed("SST_footprint_mean01")
joint = ObservationSet([sst_mean, *tracks_op.parts])
R_joint = [torch.full((sst_mean.n_obs,), 0.15**2), *R_true]
both = assimilate_general(net, cfg, joint, R_joint, instrument_noise=True, obs_seed=2, **common)

for c in names:
    m = both.samples.sel(channel=c).mean("sample").values
    print(f"{c}: ensemble-mean RMSE {rmse(m, win.truth[c]).mean():.4g}")
for g, name in enumerate(joint.names):
    sel = both.obs_group.values == g
    z = (both.predicted.values[:, sel] - both.y.values[sel]) / both.error_std.values[sel]
    print(f"  {name}: {sel.sum()} obs, residual RMS {np.sqrt((z**2).mean()):.2f} error std")
print("scaling Q per part (model units per physical unit):",
      [float(both.scaling.values[both.obs_group.values == g][0]) for g in range(len(joint.names))])

**Why the scaling matters.** The default scaling divides each observation by
its error before the solve. The alternative, `scaling="row_norm"`, makes each
operator row unit size in model units (what `assimilate` does for masks). It
works for one product, but here the SST footprints (0.15 °C errors, tiny in
model units) sit beside SSH tracks whose residuals are much larger in model
units, and a 4-iteration solve spends its effort on SSH. When we ran it, the
footprints were left about 5 error stds from the observations and SST RMSE
nearly tripled; the default fits them to about 1. Run it to see:

In [ ]:
row_norm = assimilate_general(net, cfg, joint, R_joint, instrument_noise=True, obs_seed=2,
                              scaling="row_norm", **common)
for label, ds in [("scaling='error' (default)", both), ("scaling='row_norm'", row_norm)]:
    sel = ds.obs_group.values == 0  # the SST footprints
    z = (ds.predicted.values[:, sel] - ds.y.values[sel]) / ds.error_std.values[sel]
    sst_err = rmse(ds.samples.sel(channel="SST").mean("sample").values, win.truth["SST"]).mean()
    print(f"{label:26s} SST footprint residual RMS {np.sqrt((z**2).mean()):.2f} error std, "
          f"SST RMSE {sst_err:.3f} degC")

## Caveats and checks

* **Units and scaling.** `assimilate_general` converts `H`, `y` and `R`
  together (`A_model = Q H S`, `R_model = Q R Qᵀ`). The default
  `scaling="error"` puts every observation in units of its own error, so a
  truncated MMPS solve weighs products by how informative they are, not by
  their units (section 3 shows what goes wrong otherwise). The diagonal
  preconditioner (`mmps_preconditioner="diagonal"`) or more iterations also
  help, at a cost.
* **Rozet guidance** (`guidance="rozet"`) adds its variance in observation
  space, which equals the state-space heuristic only for single-pixel
  observations; for averages and footprints MMPS is the principled choice.
* **Near-hard constraints.** Very small `R` makes the truncated solve badly
  conditioned. Keep `R` realistic (with representation error) and check
  convergence by increasing `mmps_iterations` and `steps`.
* **The residual RMS** in units of the error is a diagnostic, not a target:
  its expected value depends on the posterior uncertainty and the prior.
* **Exactness.** `tests/test_operators.py` checks that MMPS with these
  operators and correlated `R` reproduces the exact posterior of a Gaussian
  prior when the solve converges; that does not make samples from a learned
  ocean prior calibrated.
* **Not covered:** nonlinear operators, and conditional models (their
  observation inputs are mask-shaped).